# DL Math Project — AG News Classification

**Single notebook workflow:**  
1. Load & inspect data  
2. Preprocess text (`src/preprocessing.py`)  
3. Build models (`src/model_builder.py`)  
4. Train & evaluate (`src/utils.py`)  
5. Save artifacts to `models/` and `reports/`  

> **Tip:** Run all cells top-to-bottom. Each section is self-contained and prints its own outputs.

In [ ]:
# =============================================================================
# CELL 1 — Imports & Paths
# =============================================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ is on the path so we can import our helper modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from src import preprocessing, model_builder, utils

# Reproducibility
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

print("Project root:", PROJECT_ROOT)

In [ ]:
# =============================================================================
# CELL 2 — Load AG News Dataset
# =============================================================================
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

# The AG News CSV usually has no header; we name columns manually.
cols = ["class_index", "title", "description"]

df_train = pd.read_csv(train_path, names=cols)
df_test = pd.read_csv(test_path, names=cols)

# AG News labels are 1-based; convert to 0-based for Keras
df_train["label"] = df_train["class_index"] - 1
df_test["label"] = df_test["class_index"] - 1

# Combine title + description for richer text
df_train["text"] = df_train["title"].astype(str) + " " + df_train["description"].astype(str)
df_test["text"] = df_test["title"].astype(str) + " " + df_test["description"].astype(str)

CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
df_train.head()

In [ ]:
# =============================================================================
# CELL 3 — Quick EDA
# =============================================================================
print(df_train["label"].value_counts().sort_index())

plt.figure(figsize=(6, 3))
df_train["label"].value_counts().sort_index().plot(kind="bar", rot=0)
plt.xticks(range(4), CLASS_NAMES)
plt.title("Class Distribution — Training Set")
plt.ylabel("Count")
plt.show()

In [ ]:
# =============================================================================
# CELL 4 — LSTM Preprocessing
# =============================================================================
MAXLEN = 100
VOCAB_SIZE = 20000

X_train_seq, vocab = preprocessing.prepare_lstm_data(
    df_train["text"].tolist(), maxlen=MAXLEN, vocab_size=VOCAB_SIZE
)
X_test_seq, _ = preprocessing.prepare_lstm_data(
    df_test["text"].tolist(), maxlen=MAXLEN, vocab_size=0  # re-use vocab below
)

# Use the same vocab for test (manually map with the train vocab)
test_clean = preprocessing.clean_corpus(df_test["text"].tolist())
test_sequences = preprocessing.texts_to_sequences(test_clean, vocab)
X_test_seq = preprocessing.pad_sequences(test_sequences, maxlen=MAXLEN)

# One-hot encode labels
from tensorflow.keras.utils import to_categorical
y_train_oh = to_categorical(df_train["label"], num_classes=4)
y_test_oh = to_categorical(df_test["label"], num_classes=4)

print("X_train shape:", X_train_seq.shape)
print("X_test  shape:", X_test_seq.shape)
print("Vocab size:", len(vocab))

In [ ]:
# =============================================================================
# CELL 5 — Build & Train LSTM
# =============================================================================
lstm_model = model_builder.build_lstm_model(
    vocab_size=len(vocab),
    embedding_dim=128,
    maxlen=MAXLEN,
    lstm_units=64,
    dropout_rate=0.5,
    num_classes=4,
)

lstm_model.summary()

history_lstm = lstm_model.fit(
    X_train_seq, y_train_oh,
    validation_split=0.1,
    epochs=5,
    batch_size=256,
    verbose=1,
)

In [ ]:
# =============================================================================
# CELL 6 — Evaluate LSTM
# =============================================================================
lstm_preds = lstm_model.predict(X_test_seq)
lstm_pred_labels = np.argmax(lstm_preds, axis=1)

_ = utils.plot_training_history(
    history_lstm.history,
    save_path=os.path.join(PROJECT_ROOT, "reports", "lstm_training.png")
)
plt.show()

_ = utils.plot_confusion_matrix(
    df_test["label"].values,
    lstm_pred_labels,
    CLASS_NAMES,
    normalize=True,
    save_path=os.path.join(PROJECT_ROOT, "reports", "lstm_confusion_matrix.png")
)
plt.show()

report = utils.print_classification_report(
    df_test["label"].values, lstm_pred_labels, CLASS_NAMES
)

# Save model
lstm_model.save(os.path.join(PROJECT_ROOT, "models", "lstm_model.keras"))
print("\nLSTM model saved to models/lstm_model.keras")

In [ ]:
# =============================================================================
# CELL 7 — BERT Preprocessing
# =============================================================================
tokenizer = model_builder.get_bert_tokenizer("bert-base-uncased")
MAX_LEN_BERT = 128

def encode_bert(texts):
    return tokenizer(
        texts.tolist(),
        max_length=MAX_LEN_BERT,
        truncation=True,
        padding="max_length",
        return_tensors="np",
    )

train_enc = encode_bert(df_train["text"])
test_enc = encode_bert(df_test["text"])

print("BERT input IDs shape:", train_enc["input_ids"].shape)

In [ ]:
# =============================================================================
# CELL 8 — Build & Train BERT Classifier
# =============================================================================
bert_model = model_builder.build_bert_classifier(
    bert_model_name="bert-base-uncased",
    max_length=MAX_LEN_BERT,
    num_classes=4,
    learning_rate=2e-5,
)

# Optional: unfreeze top 2 transformer layers for fine-tuning
# bert_model.get_layer("tf_bert_model").bert.encoder.layer[-2:].trainable = True

bert_model.summary()

history_bert = bert_model.fit(
    {"input_ids": train_enc["input_ids"], "attention_mask": train_enc["attention_mask"]},
    y_train_oh,
    validation_split=0.1,
    epochs=3,          # Keep small because BERT fine-tuning is slow on CPU
    batch_size=32,
    verbose=1,
)

In [ ]:
# =============================================================================
# CELL 9 — Evaluate BERT
# =============================================================================
bert_preds = bert_model.predict(
    {"input_ids": test_enc["input_ids"], "attention_mask": test_enc["attention_mask"]}
)
bert_pred_labels = np.argmax(bert_preds, axis=1)

_ = utils.plot_training_history(
    history_bert.history,
    save_path=os.path.join(PROJECT_ROOT, "reports", "bert_training.png")
)
plt.show()

_ = utils.plot_confusion_matrix(
    df_test["label"].values,
    bert_pred_labels,
    CLASS_NAMES,
    normalize=True,
    save_path=os.path.join(PROJECT_ROOT, "reports", "bert_confusion_matrix.png")
)
plt.show()

report = utils.print_classification_report(
    df_test["label"].values, bert_pred_labels, CLASS_NAMES
)

# Save model
bert_model.save(os.path.join(PROJECT_ROOT, "models", "bert_model.keras"))
print("\nBERT model saved to models/bert_model.keras")

In [ ]:
# =============================================================================
# CELL 10 — Summary & Comparison
# =============================================================================
lstm_acc = utils.compute_metrics(df_test["label"].values, lstm_pred_labels)["accuracy"]
bert_acc = utils.compute_metrics(df_test["label"].values, bert_pred_labels)["accuracy"]

print("===== Final Test Accuracy =====")
print(f"LSTM : {lstm_acc:.4f}")
print(f"BERT : {bert_acc:.4f}")
print("================================")

print("\nArtifacts saved:")
print("  models/lstm_model.keras")
print("  models/bert_model.keras")
print("  reports/lstm_training.png")
print("  reports/lstm_confusion_matrix.png")
print("  reports/bert_training.png")
print("  reports/bert_confusion_matrix.png")